# Ingest to Bronze Layer
## CSV → PySpark → Delta Lake → DuckDB

### Overview
This notebook ingests raw CSV files into the Bronze layer using PySpark and Delta Lake, then loads them into DuckDB for downstream processing. Handles three separate datasets:
- **Training data** (`train-*.csv`) → `bronze.reviews_train`
- **Test data** (`test_hidden.csv`) → `bronze.reviews_test`
- **Validation data** (`validation_hidden.csv`) → `bronze.reviews_validation`

### Key Features
- **MERGE-based idempotency**: Uses Delta Lake MERGE to prevent duplicate ingestion based on Row_id
- **Separate storage**: Train/test/validation datasets stored in independent Delta tables for clean separation
- **PERMISSIVE mode**: Captures malformed rows in `_corrupt_record` column without crashing the pipeline
- **Metadata tracking**: Adds `_ingested_at`, `_load_date`, `_source_file`, and `_index` columns for lineage
- **Delta Lake format**: Provides ACID transactions and schema evolution capabilities
- **DuckDB integration**: Natively reads Delta tables using `delta_scan()`

### Technical Configuration
- PySpark 3.5.3 + Delta Lake 3.2.0
- `SPARK_LOCAL_IP` and `PYSPARK_SUBMIT_ARGS` environment variables for proper Delta integration in Jupyter
- Cross-platform support (Windows/macOS/Linux) with automatic JAVA_HOME and HADOOP_HOME detection

## 1. Import Required Libraries
Import PySpark, Delta Lake, DuckDB, and utility libraries for data processing.

In [28]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import current_timestamp, input_file_name, monotonically_increasing_id, to_date
from pyspark.sql.window import Window
from pyspark.sql.types import StructType, StructField, IntegerType, StringType, LongType, DateType, BooleanType

from delta.tables import DeltaTable
import duckdb as db
import os
import shutil

## 2. Configure Environment
Set up Spark and Delta Lake environment variables. Includes automatic Java and Hadoop detection for Windows compatibility.

In [29]:
# Spark + Delta common setup
os.environ['SPARK_LOCAL_IP'] = '127.0.0.1'
os.environ['PYSPARK_SUBMIT_ARGS'] = '--packages io.delta:delta-spark_2.12:3.2.0 pyspark-shell'

# Windows-only compatibility setup (safe no-op on macOS/Linux)
if os.name == 'nt':
    if 'JAVA_HOME' not in os.environ or not os.environ['JAVA_HOME']:
        import glob
        jdk_candidates = sorted(glob.glob(r'C:\\Program Files\\Eclipse Adoptium\\jdk-*'), reverse=True)
        if jdk_candidates:
            os.environ['JAVA_HOME'] = jdk_candidates[0]

    if 'JAVA_HOME' in os.environ and os.environ['JAVA_HOME']:
        java_bin = os.path.join(os.environ['JAVA_HOME'], 'bin')
        if java_bin not in os.environ.get('PATH', ''):
            os.environ['PATH'] = java_bin + os.pathsep + os.environ.get('PATH', '')

    hadoop_home = os.path.abspath(os.path.join(os.getcwd(), '..', '..', '.tools', 'hadoop'))
    winutils_path = os.path.join(hadoop_home, 'bin', 'winutils.exe')
    if os.path.exists(winutils_path):
        os.environ['HADOOP_HOME'] = hadoop_home
        hadoop_bin = os.path.join(hadoop_home, 'bin')
        if hadoop_bin not in os.environ.get('PATH', ''):
            os.environ['PATH'] = hadoop_bin + os.pathsep + os.environ.get('PATH', '')

print('OS:', os.name)
print('JAVA_HOME:', os.environ.get('JAVA_HOME', '<not set>'))
print('HADOOP_HOME:', os.environ.get('HADOOP_HOME', '<not set>'))

OS: posix
JAVA_HOME: <not set>
HADOOP_HOME: <not set>


## 3. Initialize Spark Session
Create a SparkSession with Delta Lake extensions enabled for ACID transaction support.

In [30]:
spark = SparkSession.builder \
    .master("local[*]") \
    .config("spark.driver.bindAddress", "127.0.0.1") \
    .config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension") \
    .config("spark.sql.catalog.spark_catalog", "org.apache.spark.sql.delta.catalog.DeltaCatalog") \
    .config("spark.sql.legacy.csv.headerCheck.enabled", "false") \
    .getOrCreate()

## 4. Define Transaction Schema
Define the expected schema for incoming CSV files. Includes `_corrupt_record` field to capture malformed rows in PERMISSIVE mode.

In [31]:
transaction_schema_training = StructType([
    StructField("Row_id", StringType(), True),
    StructField("product_id", StringType(), True),
    StructField("product_parent", StringType(), True),
    StructField("product_title", StringType(), True),
    StructField("vine", StringType(), True),
    StructField("verified_purchase", StringType(), True),
    StructField("review_headline", StringType(), True),
    StructField("review_body", StringType(), True),
    StructField("review_date", StringType(), True),
    StructField("marketplace_id", StringType(), True),
    StructField("product_category_id", LongType(), True),
    StructField("label", StringType(), True),
    
    # catches rows that fail the schema check
    StructField("_corrupt_record", StringType(), True)
])

transaction_schema_test_val = StructType([
    StructField("Row_id", StringType(), True),
    StructField("product_id", StringType(), True),
    StructField("product_parent", StringType(), True),
    StructField("product_title", StringType(), True),
    StructField("vine", StringType(), True),
    StructField("verified_purchase", StringType(), True),
    StructField("review_headline", StringType(), True),
    StructField("review_body", StringType(), True),
    StructField("review_date", StringType(), True),
    StructField("marketplace_id", StringType(), True),
    StructField("product_category_id", StringType(), True),
    
    # catches rows that fail the schema check
    StructField("_corrupt_record", StringType(), True)
])

## 5. Load CSV Data to DataFrames

### What This Step Does
Reads raw CSV files from the source directory and enriches them with metadata columns for tracking and lineage. Loads three separate datasets:
- **Training data**: `train-*.csv` files → `bronze.reviews_train`
- **Test data**: `test_hidden.csv` → `bronze.reviews_test`
- **Validation data**: `validation_hidden.csv` → `bronze.reviews_validation`

### Key Operations
1. **Path Configuration**: Sets up source (`reviews (copy)`) and three destination paths under `data/bronze/`
2. **Cleanup Check**: Removes any old non-Delta data to ensure clean Delta Lake initialization
3. **CSV Reading**: Loads each dataset with PERMISSIVE mode error handling
4. **Metadata Enrichment**: Adds four tracking columns to all datasets:
   - `_ingested_at`: Timestamp of when the row was processed
   - `_load_date`: Date partition key for efficient querying
   - `_source_file`: Full file path for data lineage
   - `_index`: Unique row identifier using `monotonically_increasing_id()`

### Error Handling
- **PERMISSIVE mode**: Malformed rows are captured in `_corrupt_record` instead of failing the entire job
- **Schema enforcement**: All rows must match the predefined `transaction_schema`
- Non-conforming data is logged but doesn't stop processing

In [32]:
# current working directory
current_dir = os.getcwd()
# project root directory (two levels up from current directory)
project_root = os.path.abspath(os.path.join(current_dir, "../../"))

# input path
raw_csv_dir = os.path.join(project_root, "reviews (copy)")
print(f"Reading from: {raw_csv_dir}")

# output paths for each dataset split
bronze_train_path = os.path.join(project_root, "data", "bronze", "train")
bronze_test_path = os.path.join(project_root, "data", "bronze", "test")
bronze_validation_path = os.path.join(project_root, "data", "bronze", "validation")

print(f"Writing train to:      {bronze_train_path}")
print(f"Writing test to:       {bronze_test_path}")
print(f"Writing validation to: {bronze_validation_path}")

# Clean up old non-Delta data if needed (first time only)
for path in [bronze_train_path, bronze_test_path, bronze_validation_path]:
    if os.path.exists(path) and not os.path.exists(os.path.join(path, "_delta_log")):
        shutil.rmtree(path)

# TRAINING DATA
base_df = (spark.read 
    # tells Spark the first line of the CSV contains the column names, not data
    .option("header", "true") 
    # if Spark finds a broken/malformed row, don't crash the script. Process it anyway.
    .option("mode", "PERMISSIVE")
    # take those broken rows from PERMISSIVE mode and stuff the raw text into this column
    .option("columnNameOfCorruptRecord", "_corrupt_record")
    # force the data to strictly match the data types and column order of this predefined schema
    .schema(transaction_schema_training)
    # only read files in the folder that specifically match this naming pattern
    .option("pathGlobFilter", "train-*.csv") 
    # point Spark to the actual folder where the CSVs live and execute the read
    .csv(raw_csv_dir)

# adding metadata
    # add a metadata column with the exact date and time this row was processed
    .withColumn("_ingested_at", current_timestamp())
    # add a metadata column with just the date (YYYY-MM-DD), great for partitioning later
    .withColumn("_load_date", to_date(current_timestamp()))
    # add a metadata column showing the exact file path this specific row came from
    .withColumn("_source_file", input_file_name())
    # assign a unique, increasing ID number to every single row Spark processes
    .withColumn("_index", monotonically_increasing_id())
)

print(f"\nTraining data loaded: {base_df.count()} rows")
base_df.select("Row_id", "product_id", "product_title").show(5)

Reading from: /home/ronlakeman/MasterInformationStudies/BigDataCourse/big-data-course-2024-projects/BigDataProject/reviews (copy)
Writing train to:      /home/ronlakeman/MasterInformationStudies/BigDataCourse/big-data-course-2024-projects/BigDataProject/data/bronze/train
Writing test to:       /home/ronlakeman/MasterInformationStudies/BigDataCourse/big-data-course-2024-projects/BigDataProject/data/bronze/test
Writing validation to: /home/ronlakeman/MasterInformationStudies/BigDataCourse/big-data-course-2024-projects/BigDataProject/data/bronze/validation

Training data loaded: 9614 rows
+------+----------+--------------------+
|Row_id|product_id|       product_title|
+------+----------+--------------------+
|     7|B00AYBGX10|Die Hüter des Lichts|
|    14|B00AW8UK0K|The Fast Diet: Th...|
|    24|B002C1BHIO|AmazonBasics Câbl...|
|    43|B005E7AOLY|"Green Naugahyde ...|
|    67|B003L0VJF6|Saint Saens: Viol...|
+------+----------+--------------------+
only showing top 5 rows



26/03/11 14:12:27 WARN CSVHeaderChecker: CSV header does not conform to the schema.
 Header: , product_id, product_title
 Schema: Row_id, product_id, product_title
Expected: Row_id but found: 
CSV file: file:///home/ronlakeman/MasterInformationStudies/BigDataCourse/big-data-course-2024-projects/BigDataProject/reviews%20(copy)/train-3.csv


In [33]:
# test data
test_df = (spark.read 
    .option("header", "true") 
    .option("mode", "PERMISSIVE")
    .option("columnNameOfCorruptRecord", "_corrupt_record")
    .schema(transaction_schema_test_val)
    .option("pathGlobFilter", "test_hidden.csv") 
    .csv(raw_csv_dir)
    .withColumn("_ingested_at", current_timestamp())
    .withColumn("_load_date", to_date(current_timestamp()))
    .withColumn("_source_file", input_file_name())
    .withColumn("_index", monotonically_increasing_id())
)
# test_df.select("Row_id", "product_id", "product_title").show(5)
print(f"Test data loaded: {test_df.count()} rows")

# validation
validation_df = (spark.read 
    .option("header", "true") 
    .option("mode", "PERMISSIVE")
    .option("columnNameOfCorruptRecord", "_corrupt_record")
    .schema(transaction_schema_test_val)
    .option("pathGlobFilter", "validation_hidden.csv") 
    .csv(raw_csv_dir)
    .withColumn("_ingested_at", current_timestamp())
    .withColumn("_load_date", to_date(current_timestamp()))
    .withColumn("_source_file", input_file_name())
    .withColumn("_index", monotonically_increasing_id())
)
# validation_df.select("Row_id", "product_id", "product_title").show(5)
print(f"Validation data loaded: {validation_df.count()} rows")

Test data loaded: 1137 rows
Validation data loaded: 1249 rows


## 6. MERGE into Delta Tables (Idempotent Upsert)
Use Delta Lake MERGE operation for all three datasets (train, test, validation):
- **Update** existing rows if `Row_id` matches
- **Insert** new rows if `Row_id` doesn't exist

This ensures idempotency - running the notebook multiple times won't create duplicates.
Each dataset is stored in a separate Delta table for clean separation.

In [34]:
# Function to perform MERGE or initial write for a dataset
def merge_or_create_table(df, path, dataset_name):
    if DeltaTable.isDeltaTable(spark, path):
        print(f"{dataset_name}: Existing Delta table found. Merging new data...")
        
        target_table = DeltaTable.forPath(spark, path)
        target_table.alias("target").merge(
            df.alias("source"),
            "target.Row_id = source.Row_id"
        ) \
        .whenMatchedUpdateAll() \
        .whenNotMatchedInsertAll() \
        .execute()
    else:
        print(f"{dataset_name}: No existing Delta table found. Creating new table...")
        df.write.format("delta").mode("overwrite").save(path)

# Merge all three datasets
merge_or_create_table(base_df, bronze_train_path, "Training")
merge_or_create_table(test_df, bronze_test_path, "Test")
merge_or_create_table(validation_df, bronze_validation_path, "Validation")
print("Done loading")

Training: No existing Delta table found. Creating new table...


26/03/11 14:12:28 WARN CSVHeaderChecker: CSV header does not conform to the schema.
 Header: , product_id, product_parent, product_title, vine, verified_purchase, review_headline, review_body, review_date, marketplace_id, product_category_id, label
 Schema: Row_id, product_id, product_parent, product_title, vine, verified_purchase, review_headline, review_body, review_date, marketplace_id, product_category_id, label
Expected: Row_id but found: 
CSV file: file:///home/ronlakeman/MasterInformationStudies/BigDataCourse/big-data-course-2024-projects/BigDataProject/reviews%20(copy)/train-5.csv
26/03/11 14:12:28 WARN CSVHeaderChecker: CSV header does not conform to the schema.
 Header: , product_id, product_parent, product_title, vine, verified_purchase, review_headline, review_body, review_date, marketplace_id, product_category_id, label
 Schema: Row_id, product_id, product_parent, product_title, vine, verified_purchase, review_headline, review_body, review_date, marketplace_id, product_cat

Test: No existing Delta table found. Creating new table...


26/03/11 14:12:29 WARN CSVHeaderChecker: CSV header does not conform to the schema.
 Header: , product_id, product_parent, product_title, vine, verified_purchase, review_headline, review_body, review_date, marketplace_id, product_category_id
 Schema: Row_id, product_id, product_parent, product_title, vine, verified_purchase, review_headline, review_body, review_date, marketplace_id, product_category_id
Expected: Row_id but found: 
CSV file: file:///home/ronlakeman/MasterInformationStudies/BigDataCourse/big-data-course-2024-projects/BigDataProject/reviews%20(copy)/test_hidden.csv


Validation: No existing Delta table found. Creating new table...


26/03/11 14:12:30 WARN CSVHeaderChecker: CSV header does not conform to the schema.
 Header: , product_id, product_parent, product_title, vine, verified_purchase, review_headline, review_body, review_date, marketplace_id, product_category_id
 Schema: Row_id, product_id, product_parent, product_title, vine, verified_purchase, review_headline, review_body, review_date, marketplace_id, product_category_id
Expected: Row_id but found: 
CSV file: file:///home/ronlakeman/MasterInformationStudies/BigDataCourse/big-data-course-2024-projects/BigDataProject/reviews%20(copy)/validation_hidden.csv


Done loading


## 7. Validate Bronze Data
Load all three Delta tables and verify the data was written correctly. Display row counts and sample data for each dataset.

In [35]:
# Load and validate all three Delta tables
train_table_df = spark.read.format("delta").load(bronze_train_path)
test_table_df = spark.read.format("delta").load(bronze_test_path)
validation_table_df = spark.read.format("delta").load(bronze_validation_path)

print(f"Total rows training data: {train_table_df.count()}")
# train_table_df.select("Row_id", "product_id", "product_title", "_load_date").show(5)

print(f"Total rows test data: {test_table_df.count()}")
# test_table_df.select("Row_id", "product_id", "product_title", "_load_date").show(5)

print(f"Total rows validation data: {validation_table_df.count()}")
# validation_table_df.select("Row_id", "product_id", "product_title", "_load_date").show(5)

Total rows training data: 9614


Total rows test data: 1137
Total rows validation data: 1249


## 8. Load Delta Tables into DuckDB
Create/replace three separate tables in DuckDB using native `delta_scan()` function:
- `bronze.reviews_train` - Training dataset
- `bronze.reviews_test` - Test dataset  
- `bronze.reviews_validation` - Validation dataset

DuckDB provides faster analytical queries and easier integration with downstream processes.

In [36]:
with db.connect(os.path.join(project_root, "ProjectData.duckdb")) as con:
    # Ensure the schema exists first!
    con.execute("CREATE SCHEMA IF NOT EXISTS bronze;")

    con.execute(f"""
        CREATE OR REPLACE TABLE bronze.reviews_train AS 
        SELECT * FROM delta_scan('{bronze_train_path}')
    """)


    # training data
    train_count = con.execute("SELECT COUNT(*) FROM bronze.reviews_train").fetchone()[0]
    train_corrupt = con.execute("SELECT COUNT(*) FROM bronze.reviews_train WHERE _corrupt_record IS NOT NULL").fetchone()[0]

    print(f"Total rows training in db: {train_count}")
    print(f"Corrupt rows training in db: {train_corrupt}")
    
    # test data
    con.execute(f"""
        CREATE OR REPLACE TABLE bronze.reviews_test AS 
        SELECT * FROM delta_scan('{bronze_test_path}')
    """)
    test_count = con.execute("SELECT COUNT(*) FROM bronze.reviews_test").fetchone()[0]
    test_corrupt = con.execute("SELECT COUNT(*) FROM bronze.reviews_test WHERE _corrupt_record IS NOT NULL").fetchone()[0]

    print(f"\nTotal rows test data in db: {test_count}")
    print(f"Corrupt rowstest data in db: {test_corrupt}")
    
    # VALIDATION DATA
    con.execute(f"""
        CREATE OR REPLACE TABLE bronze.reviews_validation AS 
        SELECT * FROM delta_scan('{bronze_validation_path}')
    """)
    validation_count = con.execute("SELECT COUNT(*) FROM bronze.reviews_validation").fetchone()[0]
    validation_corrupt = con.execute("SELECT COUNT(*) FROM bronze.reviews_validation WHERE _corrupt_record IS NOT NULL").fetchone()[0]

    print(f"\nTotal rows validation data in db: {validation_count}")
    print(f"Corrupt rows validation data in db: {validation_corrupt}")

Total rows training in db: 9614
Corrupt rows training in db: 1051

Total rows test data in db: 1137
Corrupt rowstest data in db: 122

Total rows validation data in db: 1249
Corrupt rows validation data in db: 147
